In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.api import AnchorSlice, PredictOptions
from src.app.runtime import (
    build_dataset,
    build_loader,
    load_defaults,
    load_predict_config,
    load_predictor,
)
from src.modeling.phases import quantize_phase


def take_slice(volume, axis, index):
    return np.take(volume, index, axis=axis)


In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"

In [ ]:
model_runs, predict_config = load_predict_config(PREDICT_CONFIG)
model_runs = {
    name: None
    if path is None
    else Path(path) if Path(path).is_absolute() else ROOT / path
    for name, path in model_runs.items()
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = Namespace(**load_defaults(model_runs["vae_run_dir"] / "vae.yaml"))
args.data_dir = ROOT / args.data_dir
args.batch_size = 16

batch, _ = next(
    build_loader(build_dataset(args), args, device=torch.device("cpu"))
)
target_images = [
    quantize_phase(image, args.num_phases).numpy()
    for image in batch[:, 0]
]
phase_counts = np.bincount(
    np.stack(target_images).reshape(-1),
    minlength=args.num_phases,
)
reference_phase_fractions = tuple((phase_counts / phase_counts.sum()).tolist())
anchor_image = target_images[0]
center_index = anchor_image.shape[0] // 2
anchors = [
    AnchorSlice(image=anchor_image, axis=0, index=center_index),
    AnchorSlice(image=anchor_image.copy(), axis=1, index=center_index),
]
if predict_config["phase_fractions"] is None:
    predict_config["phase_fractions"] = reference_phase_fractions
options = PredictOptions(num_phases=args.num_phases, **predict_config)

predictor = load_predictor(**model_runs, device=device)
start_time = time.perf_counter()
volume, stats = predictor.predict(
    options,
    anchors=anchors,
    target_images=target_images,
)
elapsed_seconds = time.perf_counter() - start_time
volume_np = volume.cpu().numpy()
joint_steps = int(stats["joint_history"]["step"].numel())

print(f"elapsed: {elapsed_seconds:.1f}s | volume: {volume_np.shape}")
print(f"joint steps: {joint_steps} | refine: {bool(stats['refine_applied'])}")
print(
    "phase fractions:",
    np.round(stats["final_phase_fraction"].cpu().numpy(), 4).tolist(),
)
print(
    "anchor mismatches:",
    np.round(stats["final_anchor_mismatches"].cpu().numpy(), 4).tolist(),
)

In [ ]:
anchor_mismatches = stats["final_anchor_mismatches"].cpu().numpy()
anchor_phase_mismatches = stats["final_anchor_phase_mismatches"].cpu().numpy()
axis_transition_rate = stats["final_axis_transition_rate"].cpu().numpy()
axis_run_mae = stats["final_axis_run_profile_mae"].cpu().numpy()
component_count = stats["final_component_count"].cpu().numpy()
euler_3d_density = stats["final_euler_3d_density"].cpu().numpy()
phase_axis_percolation = stats["final_phase_axis_percolation"].cpu().numpy()

print(
    "anchor phase recall:",
    np.round(1.0 - anchor_phase_mismatches, 4).tolist(),
)
print(
    "axis transition / run MAE:",
    {
        "transition": np.round(axis_transition_rate, 4).tolist(),
        "run_mae": np.round(axis_run_mae, 4).tolist(),
    },
)
print(
    "topology:",
    {
        "components": component_count.astype(int).tolist(),
        "euler_density": np.round(euler_3d_density, 6).tolist(),
        "percolation": phase_axis_percolation.astype(int).tolist(),
    },
)

fig, axes = plt.subplots(
    len(anchors), 3, figsize=(11, 3.6 * len(anchors)), squeeze=False
)
for row, anchor in enumerate(anchors):
    generated = take_slice(volume_np, anchor.axis, anchor.index)
    difference = generated != anchor.image
    panels = [
        (
            anchor.image,
            f"condition axis={anchor.axis}",
            "gray",
            0,
            args.num_phases - 1,
        ),
        (
            generated,
            f"generated index={anchor.index}",
            "gray",
            0,
            args.num_phases - 1,
        ),
        (difference, f"different {anchor_mismatches[row]:.1%}", "magma", 0, 1),
    ]
    for axis, (image, title, cmap, vmin, vmax) in zip(axes[row], panels):
        axis.imshow(
            image,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
        )
        axis.set_title(title)
        axis.axis("off")
plt.tight_layout()

offsets = np.arange(-8, 9)
fig, axes = plt.subplots(
    len(anchors),
    len(offsets),
    figsize=(25, 3.2 * len(anchors)),
    squeeze=False,
)
for row, anchor in enumerate(anchors):
    for column, offset in enumerate(offsets):
        index = anchor.index + int(offset)
        axes[row, column].imshow(
            take_slice(volume_np, anchor.axis, index),
            cmap="gray",
            vmin=0,
            vmax=args.num_phases - 1,
            interpolation="nearest",
        )
        axes[row, column].set_title(f"{offset:+d}")
        axes[row, column].axis("off")
    axes[row, 0].set_ylabel(f"axis {anchor.axis}")
fig.suptitle("Anchor neighborhoods (offset -8 to +8)")
plt.tight_layout()

indices = np.linspace(0, volume_np.shape[0] - 1, 9).round().astype(int)
fig, axes = plt.subplots(3, len(indices), figsize=(18, 6), squeeze=False)
for row, axis_id in enumerate(range(3)):
    for column, index in enumerate(indices):
        axes[row, column].imshow(
            take_slice(volume_np, axis_id, index),
            cmap="gray",
            vmin=0,
            vmax=args.num_phases - 1,
            interpolation="nearest",
        )
        axes[row, column].set_title(f"axis {axis_id} / {index}")
        axes[row, column].axis("off")
plt.tight_layout()

fig, profile_axis = plt.subplots(figsize=(10, 3.5))
for axis_id in range(3):
    adjacent_change = [
        np.mean(
            take_slice(volume_np, axis_id, index)
            != take_slice(volume_np, axis_id, index - 1)
        )
        for index in range(1, volume_np.shape[axis_id])
    ]
    profile_axis.plot(
        range(1, volume_np.shape[axis_id]),
        adjacent_change,
        label=f"axis {axis_id}",
    )
for anchor_index in sorted({anchor.index for anchor in anchors}):
    profile_axis.axvline(anchor_index, color="black", linestyle="--", alpha=0.4)
profile_axis.set(
    title="Adjacent-slice change around anchors",
    xlabel="slice index",
    ylabel="changed voxel fraction",
)
profile_axis.legend()
profile_axis.grid(alpha=0.2)
plt.tight_layout()

In [ ]:
%gui qt
import napari

try:
    viewer.close()
except NameError:
    pass

viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(
    volume_np.astype(np.int32, copy=False),
    name="prediction",
    opacity=0.8,
)
viewer.reset_view()
viewer
